# NLP Autocomplete and Autocorrect System

**Objective:** Build and evaluate autocomplete and autocorrect approaches using a real text corpus.

This notebook uses the NLTK Gutenberg corpus, frequency-based bigram/trigram autocomplete, edit-distance autocorrect, at least 10 autocomplete prefixes, 20 misspelling tests, evaluation metrics, a top-20 frequency chart, and an autocorrect confusion matrix.

Run cells from top to bottom. The corpus is downloaded on the first run. Stopwords are removed for frequency analysis; they are retained in the language-model stream because they are essential for natural sentence suggestions.

## 1. Import libraries and setup

In [ ]:
import re, sys, subprocess, importlib.util
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk

if importlib.util.find_spec("spellchecker") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspellchecker", "-q"])
    except Exception as e:
        print("pyspellchecker install skipped:", e)
try:
    from spellchecker import SpellChecker
    SPELLCHECKER_AVAILABLE = True
except Exception:
    SPELLCHECKER_AVAILABLE = False
print("pyspellchecker available:", SPELLCHECKER_AVAILABLE)

## 2. Collect a real text corpus
We use NLTK's public-domain Gutenberg collection.

In [ ]:
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg
file_ids = gutenberg.fileids()
raw_text = " ".join(gutenberg.raw(fid) for fid in file_ids)
print("Corpus files:", len(file_ids))
print("Raw characters:", len(raw_text))
print(raw_text[:500])

## 3. Preprocess text
Lowercase, remove punctuation and tokenise. We retain stopwords for autocomplete context, and create a stopword-free stream for frequency analysis.

In [ ]:
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words("english"))

def preprocess(text):
    return re.findall(r"[a-z]+", text.lower())

tokens = preprocess(raw_text)
frequency_tokens = [w for w in tokens if w not in STOP_WORDS]
print("Language-model tokens:", len(tokens))
print("Tokens after stopword removal:", len(frequency_tokens))
print(tokens[:30])

## 4. Build unigram, bigram and trigram models

In [ ]:
unigram_counts = Counter(tokens)
bigram_counts = Counter(zip(tokens[:-1], tokens[1:]))
trigram_counts = Counter(zip(tokens[:-2], tokens[1:-1]))
bigram_next = defaultdict(Counter)
trigram_next = defaultdict(Counter)
for (a, b), count in bigram_counts.items():
    bigram_next[a][b] += count
for (a, b, c), count in trigram_counts.items():
    trigram_next[(a, b)][c] += count
vocabulary = set(unigram_counts)
print("Vocabulary:", len(vocabulary), "| Bigrams:", len(bigram_counts), "| Trigrams:", len(trigram_counts))

## 5. Autocomplete functions
Bigram uses the last typed word. Trigram uses the last two words and falls back to bigram if the context is unseen.

In [ ]:
def autocomplete_bigram(prefix, top_n=3):
    words = preprocess(prefix)
    if not words: return []
    return [w for w, _ in bigram_next.get(words[-1], Counter()).most_common(top_n)]

def autocomplete_trigram(prefix, top_n=3):
    words = preprocess(prefix)
    if len(words) >= 2:
        found = trigram_next.get((words[-2], words[-1]), Counter()).most_common(top_n)
        if found: return [w for w, _ in found]
    return autocomplete_bigram(prefix, top_n)

def suggestions(prefix):
    return {
        "Prefix": prefix,
        "Bigram top 3": ", ".join(autocomplete_bigram(prefix)) or "No suggestion",
        "Trigram top 3": ", ".join(autocomplete_trigram(prefix)) or "No suggestion"
    }

## 6. Test at least 10 autocomplete prefixes

In [ ]:
test_prefixes = [
    "in the", "of the", "to the", "it is", "there is", "i am",
    "he was", "she said", "with the", "one of", "this is", "as a"
]
autocomplete_results = pd.DataFrame([suggestions(x) for x in test_prefixes])
display(autocomplete_results)

## 7. Autocomplete precision and recall

For each test prefix, one expected next word is labelled. Precision@3 is relevant returned suggestions divided by all returned suggestions. Recall@3 is relevant returned suggestions divided by expected relevant words. This is a small illustrative evaluation set, not a universal benchmark.

In [ ]:
autocomplete_gold = [
    ("in the", "world"), ("of the", "world"), ("to the", "same"),
    ("it is", "not"), ("there is", "no"), ("i am", "sure"),
    ("he was", "not"), ("she said", "that"), ("with the", "same"),
    ("one of", "the")
]

def evaluate_autocomplete(pairs, method="trigram", k=3):
    rows, hits, returned = [], 0, 0
    for prefix, expected in pairs:
        preds = (autocomplete_trigram if method == "trigram" else autocomplete_bigram)(prefix, k)
        hit = expected in preds
        hits += int(hit); returned += len(preds)
        rows.append({"prefix": prefix, "expected": expected,
                     "predictions": ", ".join(preds), "hit_at_3": hit})
    precision = hits / returned if returned else 0
    recall = hits / len(pairs) if pairs else 0
    return pd.DataFrame(rows), precision, recall

bigram_eval, bigram_precision, bigram_recall = evaluate_autocomplete(autocomplete_gold, "bigram")
trigram_eval, trigram_precision, trigram_recall = evaluate_autocomplete(autocomplete_gold, "trigram")
print("Bigram Precision@3:", round(bigram_precision, 3), "Recall@3:", round(bigram_recall, 3))
print("Trigram Precision@3:", round(trigram_precision, 3), "Recall@3:", round(trigram_recall, 3))
display(trigram_eval)

## 8. Autocorrect approach 1: pyspellchecker

In [ ]:
spell = SpellChecker() if SPELLCHECKER_AVAILABLE else None

def pyspell_correct(word):
    if spell is None: return word
    return spell.correction(word) or word

## 9. Autocorrect approach 2: custom Levenshtein distance
Levenshtein distance counts insertions, deletions and substitutions. Among closest corpus words, the most frequent candidate is selected.

In [ ]:
def levenshtein_distance(a, b):
    if len(a) < len(b): a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(
                current[j-1] + 1,
                previous[j] + 1,
                previous[j-1] + (ca != cb)
            ))
        previous = current
    return previous[-1]

def custom_autocorrect(word, max_distance=2):
    word = re.sub(r"[^a-z]", "", word.lower())
    if not word or word in vocabulary: return word
    candidates = [w for w in vocabulary if abs(len(w)-len(word)) <= max_distance]
    best_distance, best = max_distance + 1, []
    for candidate in candidates:
        d = levenshtein_distance(word, candidate)
        if d < best_distance: best_distance, best = d, [candidate]
        elif d == best_distance: best.append(candidate)
    if not best or best_distance > max_distance: return word
    return max(best, key=lambda w: unigram_counts[w])

## 10. Test at least 20 misspellings

In [ ]:
misspelling_tests = [
    ("teh","the"), ("recieve","receive"), ("adress","address"),
    ("becuase","because"), ("thier","their"), ("goverment","government"),
    ("enviroment","environment"), ("seperate","separate"),
    ("definately","definitely"), ("accomodate","accommodate"),
    ("begining","beginning"), ("beleive","believe"),
    ("freind","friend"), ("writting","writing"),
    ("langauge","language"), ("untill","until"),
    ("wich","which"), ("throught","through"),
    ("peopel","people"), ("importent","important")
]

def evaluate_corrector(cases, corrector):
    rows = []
    for wrong, expected in cases:
        predicted = corrector(wrong)
        rows.append({"misspelling": wrong, "expected": expected,
                     "predicted": predicted, "correct": predicted == expected})
    result = pd.DataFrame(rows)
    return result, result["correct"].mean()

custom_results, custom_accuracy = evaluate_corrector(misspelling_tests, custom_autocorrect)
display(custom_results)
print("Custom Levenshtein accuracy:", round(custom_accuracy, 3))

## 11. Compare autocorrect methods

For one prediction per test word, exact-match precision and recall are calculated as correct predictions divided by all predictions/test cases. This is a micro-averaged word-level evaluation; incorrect corrections count as false positives and missed expected corrections as false negatives.

In [ ]:
comparison_rows = [{
    "Method": "Custom Levenshtein", "Accuracy": custom_accuracy,
    "Precision": custom_accuracy, "Recall": custom_accuracy
}]
if SPELLCHECKER_AVAILABLE:
    spell_results, spell_accuracy = evaluate_corrector(misspelling_tests, pyspell_correct)
    comparison_rows.append({
        "Method": "pyspellchecker", "Accuracy": spell_accuracy,
        "Precision": spell_accuracy, "Recall": spell_accuracy
    })
    display(spell_results)
else:
    spell_accuracy = np.nan
    print("pyspellchecker unavailable; custom method is still evaluated.")
autocorrect_comparison = pd.DataFrame(comparison_rows)
display(autocorrect_comparison)

## 12. Top 20 frequent words (stopwords removed)

In [ ]:
top20_df = pd.DataFrame(Counter(frequency_tokens).most_common(20),
                        columns=["Word", "Frequency"])
plt.figure(figsize=(12, 6))
plt.bar(top20_df["Word"], top20_df["Frequency"])
plt.title("Top 20 Most Frequent Words (Stopwords Removed)")
plt.xlabel("Word"); plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()
display(top20_df)

## 13. Confusion matrix for custom autocorrect

In [ ]:
labels = sorted(set(custom_results["expected"]) | set(custom_results["predicted"]))
confusion = pd.crosstab(
    pd.Categorical(custom_results["expected"], categories=labels),
    pd.Categorical(custom_results["predicted"], categories=labels),
    rownames=["Actual"], colnames=["Predicted"], dropna=False
)
plt.figure(figsize=(11, 9))
plt.imshow(confusion.values, aspect="auto")
plt.title("Custom Autocorrect Confusion Matrix")
plt.xlabel("Predicted word"); plt.ylabel("Actual word")
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.colorbar(label="Count"); plt.tight_layout(); plt.show()
display(confusion)

## 14. Compare approaches and limitations

### Autocomplete
- **Bigram:** simple and fast, but uses only the previous word.
- **Trigram:** uses two previous words and can model more context, but has more unseen contexts and may fall back to bigram.

### Autocorrect
- **pyspellchecker:** ready-made frequency dictionary and candidate generation.
- **Custom Levenshtein:** easy to explain and customise, but vocabulary search can be slow and the most frequent candidate is not always the intended word.

### Limitations compared with Google Keyboard
This project uses corpus frequencies and edit distance. A production keyboard may use personal typing history, context-aware/neural language models, user vocabulary, multilingual support, latency optimisation and privacy-aware on-device processing. Gutenberg text does not represent everyday messages, names, slang, regional English or every typing style. The test set is small and illustrative.

## 15. Final summary

In [ ]:
summary = {
    "Corpus files": len(file_ids),
    "Corpus tokens": len(tokens),
    "Vocabulary size": len(vocabulary),
    "Bigram precision@3": bigram_precision,
    "Bigram recall@3": bigram_recall,
    "Trigram precision@3": trigram_precision,
    "Trigram recall@3": trigram_recall,
    "Custom autocorrect accuracy": custom_accuracy,
    "pyspellchecker accuracy": spell_accuracy
}
display(pd.Series(summary, name="Value").to_frame())

## 16. Export result files

In [ ]:
autocomplete_results.to_csv("autocomplete_test_results.csv", index=False)
custom_results.to_csv("custom_autocorrect_results.csv", index=False)
autocorrect_comparison.to_csv("autocorrect_method_comparison.csv", index=False)
print("CSV result files saved in the notebook's current working directory.")